![image_1781185782795.png](./image_1781185782795.png "image_1781185782795.png")

In [0]:
%pip install langgraph databricks_langchain

In [0]:
%pip install langgraph --upgrade
dbutils.library.restartPython()

In [0]:
import os
import random
from dotenv import load_dotenv

from typing import Literal, TypedDict
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated
from databricks_langchain import ChatDatabricks
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition


import mlflow
mlflow.langchain.autolog()

class State(MessagesState):
    pass


def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    """Example: >>> multiply(2, 3)
    6"""
    """Args:
    a: int
    b: int"""
    """Returns:
    int"""
    return a * b


llm = ChatDatabricks(model = "databricks-gpt-oss-120b")

llm_with_tools = llm.bind_tools([multiply])

def tool_calling_llm(state: State):
    return {
        "messages": [llm_with_tools.invoke(state["messages"])]
    }


builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_edge(START, "tool_calling_llm")
builder.add_edge("tool_calling_llm", END)
graph = builder.compile() 


result = graph.invoke({
    "messages": HumanMessage(content="Hello, what is the weather in Tokyo? and what is 2 * 3?")
})


for i, message in enumerate(result["messages"]):
    print(f"Message {i+1}: ({type(message).__name__}): {message.content}")



LangChain inbuilt tools https://docs.langchain.com/oss/python/integrations/providers/overview

In [0]:
dbutils.library.restartPython()


In [0]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
 

In [0]:
import requests
from typing import Literal, TypedDict
from langchain_core.messages import AnyMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from typing_extensions import Annotated
from databricks_langchain import ChatDatabricks
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

import mlflow
mlflow.langchain.autolog()

class State(MessagesState):
    pass



@tool
def wikipedia_search(query: str) -> str:
    """Search Wikipedia and return a summary."""
    url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + query.replace(" ", "%20")

    headers = {
        "User-Agent": "databricks-agent/1.0"
    }

    try:
        r = requests.get(url, headers=headers, timeout=5)

        if r.status_code != 200:
            return f"Failed to fetch Wikipedia data: {r.status_code}"

        data = r.json()
        return data.get("extract", "No summary found.")

    except Exception as e:
        return f"Wikipedia error: {str(e)}"




#wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

llm = ChatDatabricks(endpoint = "databricks-gpt-oss-120b")
llm_with_tools = llm.bind_tools([wikipedia_search])

def tool_calling_llm(state: State):
    return {
        "messages": [llm_with_tools.invoke(state["messages"])]
    }


builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode([wikipedia_search]))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)
builder.add_edge("tools", "tool_calling_llm")

graph = builder.compile()

result = graph.invoke({
    "messages": HumanMessage("Tell me about databricks.")
})


for i, message in enumerate(result["messages"]):
    print(f"Message {i+1}: ({type(message).__name__}): {message.content}")
